In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Configuration
os.makedirs("../models", exist_ok=True)
os.makedirs("../data/output", exist_ok=True)
INPUT_PATH = "../data/processed/dvf_processed.parquet"

def train_pipeline():
    start_time = time.time()
    
    # Chargement
    df = pd.read_parquet(INPUT_PATH)
    
    # 2. Split Train/Test
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    
    # 3. Calcul des variables communales (sur le train uniquement)
    k = 15
    stats = train_df.groupby("code_commune").agg(
        commune_prix_m2=("prix_m2", "median"),
        commune_volume=("valeur_fonciere", "count")
    ).reset_index()
    
    global_median = train_df["prix_m2"].median()

    # Fonction pour éviter le KeyError : vérifie et assigne
    def add_commune_features(data, stats_df, global_val):
        data = data.merge(stats_df, on="code_commune", how="left")
        data["commune_prix_m2"] = data["commune_prix_m2"].fillna(global_val)
        data["commune_volume"] = data["commune_volume"].fillna(0)
        # Lissage
        data["commune_prix_m2"] = (data["commune_volume"] * data["commune_prix_m2"] + k * global_val) / (data["commune_volume"] + k)
        return data

    train_df = add_commune_features(train_df, stats, global_median)
    test_df = add_commune_features(test_df, stats, global_median)
    
    # 4. Cible : Résiduel
    train_df["target"] = train_df["prix_m2"] - train_df["commune_prix_m2"]
    test_df["target"] = test_df["prix_m2"] - test_df["commune_prix_m2"]
    
    # 5. Préparation X et y
    drop_cols = ["valeur_fonciere", "prix_m2", "target", "id_mutation", "date_mutation"]
    X_train = train_df.drop(columns=drop_cols)
    y_train = train_df["target"]
    X_test = test_df.drop(columns=drop_cols)
    y_test = test_df["target"]

    # CORRECTION : Convertir les colonnes 'object' en 'category' pour XGBoost
    for col in X_train.select_dtypes(include=['object']).columns:
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

    # 6. Entraînement avec enable_categorical=True
    print("Entraînement XGBoost avec support catégorie...")
    model = xgb.XGBRegressor(
        n_estimators=1000, 
        learning_rate=0.05, 
        max_depth=6, 
        n_jobs=-1, 
        random_state=42,
        enable_categorical=True  # <--- CRUCIAL
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)
    
    # 7. Exports CSV et PKL
    X_train.to_csv("../data/output/X_train.csv", index=False)
    X_test.to_csv("../data/output/X_test.csv", index=False)
    y_train.to_csv("../data/output/y_train.csv", index=False)
    y_test.to_csv("../data/output/y_test.csv", index=False)
    joblib.dump(model, "../models/xgb_model.pkl")
    
    # 8. Évaluation
    preds = model.predict(X_test) + test_df["commune_prix_m2"]
    r2 = r2_score(test_df["prix_m2"], preds)
    mae = mean_absolute_error(test_df["prix_m2"], preds)
    mape = np.mean(np.abs((test_df["prix_m2"] - preds) / test_df["prix_m2"])) * 100
    
    print(f"\n--- RÉSULTATS ---")
    print(f"R²: {r2:.4f} | MAE: {mae:.2f} €/m² | MAPE: {mape:.2f} % | Temps: {time.time()-start_time:.1f}s")

if __name__ == "__main__":
    train_pipeline()

Entraînement XGBoost avec support catégorie...
[0]	validation_0-rmse:1285.77466
[100]	validation_0-rmse:1128.85350
[200]	validation_0-rmse:1123.06149
[300]	validation_0-rmse:1122.35898
[400]	validation_0-rmse:1121.76167
[500]	validation_0-rmse:1120.62274
[600]	validation_0-rmse:1120.58639
[700]	validation_0-rmse:1119.13830
[800]	validation_0-rmse:1118.77610
[900]	validation_0-rmse:1118.75015
[999]	validation_0-rmse:1118.19425

--- RÉSULTATS ---
R²: 0.7444 | MAE: 744.36 €/m² | MAPE: 36.40 % | Temps: 233.3s
